# Rotten Fruit Detector — YOLO26 (custom container)

Fine-tunes **YOLO26** (Ultralytics, PyTorch) via a custom SageMaker container you build
and push yourself (see `docker/` — not covered by this notebook). YOLO26 isn't in
SageMaker JumpStart's built-in model zoo, so this bypasses JumpStart entirely and runs a
bring-your-own-container training job with a generic `Estimator`.

Flow: download a Roboflow **YOLOv8** (Ultralytics) format export → upload it to S3
as-is (no conversion needed) → `.fit()` launches an ephemeral GPU training instance
running your image's `train` entrypoint → deploy an endpoint running your image's
`serve` entrypoint → predict.

Expected `dataset.zip` (Roboflow **YOLOv8** export — Ultralytics' format, works for
YOLO26 too):

```text
train/images/*.jpg + train/labels/*.txt
valid/images/*.jpg + valid/labels/*.txt
test/images/*.jpg  + test/labels/*.txt
data.yaml
```

**Before running the training cell**, build your image and push it to ECR, then paste
the resulting URI into `IMAGE_URI` below.

Run on any small kernel (e.g. `ml.t3.medium`) — the heavy GPU work happens on the
separate training instance, not this notebook.

In [ ]:
# 1. Setup: SageMaker session, role, bucket
%pip install -q -r ../requirements.txt

import sagemaker
from pathlib import Path

session = sagemaker.Session()
role = sagemaker.get_execution_role()
bucket = session.default_bucket()
PROJECT_DIR = Path.cwd().parent
print("Role:", role)
print("Bucket:", bucket)

In [ ]:
# 2. Get the raw dataset from S3 and unzip it
import zipfile
import boto3

S3_DATA_ZIP = "s3://YOUR_BUCKET/datasets/dataset.zip"  # your zipped Roboflow YOLOv8-format export

b, key = S3_DATA_ZIP.replace("s3://", "").split("/", 1)
zip_path = PROJECT_DIR / "dataset.zip"
boto3.client("s3").download_file(b, key, str(zip_path))

EXTRACT_DIR = PROJECT_DIR / "dataset"
with zipfile.ZipFile(zip_path) as archive:
    archive.extractall(EXTRACT_DIR)

def find_dataset_root(base):
    base = Path(base)
    for path in [base, *(p for p in base.rglob("*") if p.is_dir())]:
        if (path / "data.yaml").exists():
            return path
    raise FileNotFoundError(f"No 'data.yaml' found under {base}")

DATA_DIR = find_dataset_root(EXTRACT_DIR)
print("Dataset root:", DATA_DIR)

In [ ]:
# 3. Upload the dataset root (Roboflow's YOLO export, as-is) to S3 (trailing slash required)
training_s3_uri = session.upload_data(
    path=str(DATA_DIR), bucket=bucket, key_prefix="rotten-fruit/yolo-input"
)
if not training_s3_uri.endswith("/"):
    training_s3_uri += "/"
print("Training data:", training_s3_uri)

In [ ]:
# 4. Fine-tune YOLO26 on an ephemeral GPU instance, using your own container image
#    (build + push it via docker/ first -- see that folder for the build/push steps).
#    ml.g5.2xlarge (A10G 24GB GPU, 32GB RAM). Needs its 'for training job usage' quota >= 1.
from train import train

IMAGE_URI = "<account-id>.dkr.ecr.<region>.amazonaws.com/rotten-fruit-yolo26:latest"  # from your docker build/push

estimator = train(
    training_s3_uri=training_s3_uri,
    role=role,
    image_uri=IMAGE_URI,
    output_path=f"s3://{bucket}/rotten-fruit/output",
    instance_type="ml.g5.2xlarge",
    epochs=75,
    batch_size=16,
    learning_rate=0.001,
    optimizer="SGD",
)

In [ ]:
# 5. Deploy an endpoint and run a prediction, then clean up
from train import deploy
from predict import predict_image

predictor = deploy(estimator, instance_type="ml.m5.xlarge")

test_image = sorted((DATA_DIR / "test" / "images").glob("*.jpg"))[0]
for det in predict_image(predictor, test_image)[:10]:
    print(det)

# IMPORTANT: delete the endpoint so it stops incurring charges.
predictor.delete_endpoint()